Load Pre process Data Into New Note Book

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [10]:
import time
import numpy as np
import pandas as pd
import requests
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

URL = "https://api.opentopodata.org/v1/srtm30m"

# quick test: one point in Walapane area
r = requests.get(URL, params={"locations": "7.0,80.7"}, timeout=60)
print(r.status_code, r.json())

200 {'results': [{'dataset': 'srtm30m', 'elevation': 1481.0, 'location': {'lat': 7.0, 'lng': 80.7}}], 'status': 'OK'}


elevation helper

In [11]:
def get_elevations(coords, batch=100, pause=1.1):
    """coords: list of (lat, lon). Returns list of elevations (m)."""
    out = []
    for i in range(0, len(coords), batch):
        chunk = coords[i:i + batch]
        loc = "|".join(f"{la:.6f},{lo:.6f}" for la, lo in chunk)
        for attempt in range(5):
            resp = requests.post(URL, json={"locations": loc, "interpolation": "bilinear"}, timeout=120)
            if resp.status_code == 200:
                out.extend([x["elevation"] for x in resp.json()["results"]])
                break
            time.sleep(3 * (attempt + 1))   # wait, then retry (rate limit)
        else:
            raise RuntimeError(f"Failed batch starting at {i}: {resp.status_code} {resp.text[:200]}")
        time.sleep(pause)                   # free API allows ~1 call per second
    return out

build the 3×3 neighbours and get elevations

In [12]:
df = pd.read_csv(DATA_PROCESSED / "landslide_dynamic_ml_dataset.csv")

STEP = 30.0   # metres between neighbouring samples
m_per_deg = 111320.0

# unique points only, to save API calls
pts = df[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
lat = pts["lat"].to_numpy()
lon = pts["lon"].to_numpy()
dlat = STEP / m_per_deg
dlon = STEP / (m_per_deg * np.cos(np.radians(lat)))

# grid positions: rows = north to south, cols = west to east
offsets = [(+1, -1), (+1, 0), (+1, +1),
           ( 0, -1), ( 0, 0), ( 0, +1),
           (-1, -1), (-1, 0), (-1, +1)]

coords = []
for k in range(len(pts)):
    for di, dj in offsets:
        coords.append((lat[k] + di * dlat, lon[k] + dj * dlon[k]))

print("Points:", len(pts), "| API coordinates:", len(coords))
elev = np.array(get_elevations(coords), dtype="float64").reshape(len(pts), 9)

Points: 955 | API coordinates: 8595


slope and elevation, then save

In [13]:
a, b, c, d, e, f, g, h, i = [elev[:, j] for j in range(9)]   # a..i = 3x3 window

dz_dx = ((c + 2*f + i) - (a + 2*d + g)) / (8 * STEP)
dz_dy = ((a + 2*b + c) - (g + 2*h + i)) / (8 * STEP)
slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy)))

pts["elevation_m"] = e
pts["slope_deg"] = slope_deg

df_terrain = df.merge(pts, on=["lat", "lon"], how="left", validate="many_to_one")
df_terrain.to_csv(DATA_PROCESSED / "landslide_dynamic_ml_dataset_terrain.csv", index=False)

print(df_terrain[["elevation_m", "slope_deg"]].describe().round(1))
print(df_terrain.groupby("dynamic_label")[["elevation_m", "slope_deg"]].median().round(1))
print("Missing:", df_terrain[["elevation_m", "slope_deg"]].isna().sum().to_dict())

       elevation_m  slope_deg
count        958.0      958.0
mean        1201.1       16.9
std          445.6        7.9
min          110.0        0.0
25%          871.0       11.5
50%         1164.0       16.1
75%         1519.0       22.3
max         2449.0       50.2
               elevation_m  slope_deg
dynamic_label                        
0                   1340.0       14.7
1                   1029.0       17.9
Missing: {'elevation_m': 0, 'slope_deg': 0}
